<a href="https://colab.research.google.com/github/Khushmandeep-Kaur-Dhaliwal/ml-project-online-shopping-data-analysis/blob/main/online_shopping_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/file.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,Unnamed: 0,CustomerID,Gender,Location,Tenure_Months,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,...,Avg_Price,Delivery_Charges,Coupon_Status,GST,Date,Offline_Spend,Online_Spend,Month,Coupon_Code,Discount_pct
0,0,17850.0,M,Chicago,12.0,16679.0,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,...,153.71,6.5,Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
1,1,17850.0,M,Chicago,12.0,16680.0,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,...,153.71,6.5,Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
2,2,17850.0,M,Chicago,12.0,16696.0,2019-01-01,GGOENEBQ078999,Nest Cam Outdoor Security Camera - USA,Nest-USA,...,122.77,6.5,Not Used,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
3,3,17850.0,M,Chicago,12.0,16699.0,2019-01-01,GGOENEBQ079099,Nest Protect Smoke + CO White Battery Alarm-USA,Nest-USA,...,81.50,6.5,Clicked,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0
4,4,17850.0,M,Chicago,12.0,16700.0,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,...,153.71,6.5,Clicked,0.1,1/1/2019,4500.0,2424.5,1,ELEC10,10.0


In [ ]:
# Get a concise summary of the DataFrame, including data types and non-null values
df.info()

# Check for missing values in each column
missing_values = df.isnull().sum()
print("\nMissing values per column:")
print(missing_values[missing_values > 0])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52955 entries, 0 to 52954
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           52955 non-null  int64  
 1   CustomerID           52924 non-null  float64
 2   Gender               52924 non-null  object 
 3   Location             52924 non-null  object 
 4   Tenure_Months        52924 non-null  float64
 5   Transaction_ID       52924 non-null  float64
 6   Transaction_Date     52924 non-null  object 
 7   Product_SKU          52924 non-null  object 
 8   Product_Description  52924 non-null  object 
 9   Product_Category     52955 non-null  object 
 10  Quantity             52924 non-null  float64
 11  Avg_Price            52924 non-null  float64
 12  Delivery_Charges     52924 non-null  float64
 13  Coupon_Status        52924 non-null  object 
 14  GST                  52924 non-null  float64
 15  Date                 52924 non-null 

In [ ]:
# Drop the 'Unnamed: 0' column as it appears to be an artifact
df = df.drop('Unnamed: 0', axis=1)

# Convert 'Transaction_Date' and 'Date' to datetime objects
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Date'] = pd.to_datetime(df['Date'])

# Impute missing numerical values with the median
for col in ['CustomerID', 'Tenure_Months', 'Transaction_ID', 'Quantity', 'Avg_Price', 'Delivery_Charges', 'GST', 'Offline_Spend', 'Online_Spend', 'Discount_pct']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# Impute missing categorical values with the mode
for col in ['Gender', 'Location', 'Product_SKU', 'Product_Description', 'Coupon_Status', 'Coupon_Code']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after imputation:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDataFrame info after cleaning:")
df.info()

Missing values after imputation:
Transaction_Date    31
Date                31
dtype: int64

DataFrame info after cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52955 entries, 0 to 52954
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   CustomerID           52955 non-null  float64       
 1   Gender               52955 non-null  object        
 2   Location             52955 non-null  object        
 3   Tenure_Months        52955 non-null  float64       
 4   Transaction_ID       52955 non-null  float64       
 5   Transaction_Date     52924 non-null  datetime64[ns]
 6   Product_SKU          52955 non-null  object        
 7   Product_Description  52955 non-null  object        
 8   Product_Category     52955 non-null  object        
 9   Quantity             52955 non-null  float64       
 10  Avg_Price            52955 non-null  float64       
 11  Delivery_Charges     

In [ ]:
# Drop rows where 'Transaction_Date' or 'Date' are missing
df.dropna(subset=['Transaction_Date', 'Date'], inplace=True)

print("Missing values after dropping rows with missing dates:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDataFrame info after dropping rows with missing dates:")
df.info()

# Define target and features
# Assuming 'Online_Spend' is the target variable to predict
target = 'Online_Spend'
features = [col for col in df.columns if col not in [target, 'Transaction_Date', 'Date', 'Product_Description', 'Product_SKU']]

X = df[features]
y = df[target]

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(exclude=['object']).columns

print(f"\nCategorical features: {list(categorical_features)}")
print(f"Numerical features: {list(numerical_features)}")

# One-hot encode categorical features
X = pd.get_dummies(X, columns=categorical_features, drop_first=True)

print("\nShape of data after one-hot encoding:", X.shape)
print(X.head())

Missing values after dropping rows with missing dates:
Series([], dtype: int64)

DataFrame info after dropping rows with missing dates:
<class 'pandas.core.frame.DataFrame'>
Index: 52924 entries, 0 to 52923
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   CustomerID           52924 non-null  float64       
 1   Gender               52924 non-null  object        
 2   Location             52924 non-null  object        
 3   Tenure_Months        52924 non-null  float64       
 4   Transaction_ID       52924 non-null  float64       
 5   Transaction_Date     52924 non-null  datetime64[ns]
 6   Product_SKU          52924 non-null  object        
 7   Product_Description  52924 non-null  object        
 8   Product_Category     52924 non-null  object        
 9   Quantity             52924 non-null  float64       
 10  Avg_Price            52924 non-null  float64       
 11  Delivery_Charg

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nLinear Regression Model Evaluation:")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

X_train shape: (42339, 80)
X_test shape: (10585, 80)
y_train shape: (42339,)
y_test shape: (10585,)

Linear Regression Model Evaluation:
Mean Squared Error (MSE): 579402.34
R-squared (R2): 0.13
